In [ ]:
# given x covariates and y pressure variables,
# fit minimum distance classifier on all variables,
# predict closest class and then 
# compute iou for each class (reconstruction accuracy- represents baseline)
# iteratively drop a variable until one remains, compute miou and keep track of miou and train covariates at each step

In [1]:
import ee

ee.Authenticate(auth_mode='notebook')
ee.Initialize(project = 'ee-gsingh')

import geemap
from geeml.utils import eeprint
import geopandas as gpd
import pandas as pd
import numpy as np
import json
from shapely import wkt
from tqdm.auto import tqdm
import os
import time
from functools import wraps

## Step 1: load covariates

In [18]:
AEF = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

AEF22 = AEF.filterDate("2022-01-01", "2023-01-01").filterBounds(ecoregion.geometry()).mosaic()

In [19]:
# Hunter et al., 2025. 2000-2022

h2000 = ee.ImageCollection("projects/global-pasture-watch/assets/gsvh-30m/v1/short-veg-height_m").filterDate('2000').first()
h2022 = ee.ImageCollection("projects/global-pasture-watch/assets/gsvh-30m/v1/short-veg-height_m").filterDate('2022').first()

# Lang et al., 2023. 2020
canopy_height = ee.Image("users/nlang/ETH_GlobalCanopyHeight_2020_10m_v1")
# combine short height (<10m) from hunter et al., 2025 with canopy height (>10m) from lang et al., 2023
cheight22 = h2022.updateMask(h2022.lte(10)).unmask(canopy_height).rename('comp_height_2022').toFloat()

# Isik, M. S., Parente, L., Consoli, D., Sloat, L., Mesquita, V. V., Ferreira, L. G., Sabbatini, S., Stanimirova, R., Teles, N. M., Robinson, N., Costa Junior, C., & Hengl, T. (2025).
# Light use efficiency (LUE) based bimonthly gross primary productivity (GPP) for global grasslands at 30 m spatial resolution (2000–2022).
# PeerJ, 13, e19774. https://doi.org/10.7717/peerj.19774

ugpp_m2000 = ee.ImageCollection("projects/global-pasture-watch/assets/ggpp-30m/v1/ugpp_m").filterDate('2000').first()
ugpp_m2022 = ee.ImageCollection("projects/global-pasture-watch/assets/ggpp-30m/v1/ugpp_m").filterDate('2022').first()


# Define the ImageCollections and mask
bii8km = ee.ImageCollection("projects/ee-geethensingh/assets/Hayley/BII_8km")
bii1km = ee.ImageCollection("projects/ee-geethensingh/assets/Hayley/BII_1km")
mask = ee.Image("projects/ee-geethensingh/assets/BIIMask")

# Define the band names
bands1km = ['Land Use', 'Land Use Intensity', 'BII All',
            'BII Amphibians', 'BII Birds', 'BII Forbs', 'BII Graminoids',
            'BII Mammals', 'BII All Plants', 'BII Reptiles', 'BII Trees',
            'BII All Vertebrates']

bands8km = ['BII All',
            'BII Amphibians', 'BII Birds', 'BII Forbs', 'BII Graminoids',
            'BII Mammals', 'BII All Plants', 'BII Reptiles', 'BII Trees',
            'BII All Vertebrates', 'Land Use', 'Land Use Intensity']

# Rename bands for 1km data
bii1km = bii1km.toBands().rename(bands1km)

# Remove no data pixels for 1km data
biionekm = bii1km.select('^BII.*').selfMask()

# Remove land uses (plantation and protected area) with no intensity for 1km data
lcMask = bii1km.select('Land Use').neq(2).And(
    bii1km.select('Land Use').neq(5))
LUI = bii1km.select('Land Use Intensity').updateMask(lcMask)

# Merge datasets for 1km data
bii1km = biionekm.addBands([bii1km.select('Land Use'), LUI]).updateMask(mask)

covariates = cheight22.rename('structure').addBands([ugpp_m2022.rename('function'), bii1km.select('BII All').rename('composition')])


In [20]:
covariates = AEF22

In [21]:
sanlc22 = ee.Image("projects/ee-gsingh/assets/RECOVER/sanlc2022_7class").rename('nlc22')
reclass = sanlc22.remap(ee.List.sequence(1, 7), ee.List([0,0,0,1,2,3,4])).rename('nlc22_reclass')
Map = geemap.Map(center=[-30, 25], zoom=5)
Map.addLayer(sanlc22, {'min':1, 'max':7, 'palette':["#7c7c73","#868987","#a2a8a9",'#2c7fb8','#253494','#081d58','#081d58']}, 'original', True)
Map.addLayer(reclass, {'min':0, 'max':4, 'palette':["#7c7c73",'#2c7fb8','#253494','#081d58','#081d58']}, 'reclass', True)
Map

Map(center=[-30, 25], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', tra…

## Implement recursive feature elimination

In [22]:
# import lsib and filter to south africa
sa = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017").filter(ee.Filter.eq('country_na', 'South Africa'))

# Ecoregions
ecoregions = ee.FeatureCollection("RESOLVE/ECOREGIONS/2017").filterBounds(sa.geometry())
econames = ecoregions.aggregate_array('ECO_NAME')

listid = 6
econame = ecoregions.aggregate_array('ECO_NAME').get(listid).getInfo()
print(f"Selecting {econame} ecoregion")
ecoregion = ecoregions.filter(ee.Filter.eq('ECO_NAME', econame))

fscs = ee.FeatureCollection("projects/ee-gsingh/assets/RECOVER/fscs_samples_v1").filterBounds(ecoregion.geometry())
eeprint(fscs.size())
eeprint(fscs.first())
train = reclass.reduceRegions(collection = fscs, reducer = ee.Reducer.first(), scale = reclass.projection().nominalScale())\
    .filter(ee.Filter.notNull(['first', 'compositio']))

Selecting Highveld grasslands ecoregion


In [23]:
train = AEF22.addBands(reclass).reduceRegions(collection = fscs, reducer = ee.Reducer.first(), scale = reclass.projection().nominalScale())

In [29]:
train.first()

In [24]:
# Load South Africa polygon
sa = ee.FeatureCollection("USDOS/LSIB_SIMPLE/2017") \
        .filter(ee.Filter.eq('country_na', 'South Africa'))

# Create grid over SA
grid = sa.geometry().coveringGrid(sa.geometry().projection(), 50000)

# Function to calculate % of cell overlapping SA
def cell_land_fraction(cell):
    cell_geom = ee.Feature(cell).geometry()
    
    # Intersection area
    intersect_area = cell_geom.intersection(sa.geometry(), 10).area(10)
    cell_area = cell_geom.area(10)
    
    # Fraction in %
    fraction = intersect_area.divide(cell_area).multiply(100)
    
    return ee.Feature(cell).set({'land_frac': fraction})

# Map over grid
grid_with_frac = ee.FeatureCollection(grid).map(cell_land_fraction)

# Filter cells with >= 50% land
grid_filtered = grid_with_frac.filter(ee.Filter.gte('land_frac', 50))

In [ ]:
n_classes = 2 # even though there are five classes, we are only really interested in seperating transformed from natural.
# if going sub transformed class then that is useful to identify errors pressures-wise
classifier = ee.Classifier.minimumDistance(metric = 'mahalanobis').train(
    features = train,
    classProperty = 'first',
    inputProperties = ['structure', 'function', 'compositio']).setOutputMode('CLASSIFICATION')
classified = covariates.rename(['structure', 'function', 'compositio']).classify(classifier)

pred_masks = classified.gte(1).eq(ee.Image.constant(ee.List.sequence(0, n_classes - 1)))
true_masks = reclass.gte(1).eq(ee.Image.constant(ee.List.sequence(0, n_classes - 1)))
# eeprint(pred_masks)

In [45]:
n_classes = 2 # even though there are five classes, we are only really interested in seperating transformed from natural.
# if going sub transformed class then that is useful to identify errors pressures-wise
classifier = ee.Classifier.minimumDistance(metric = 'cosine').train(
    features = train,
    classProperty = 'nlc22_reclass',
    inputProperties = AEF22.bandNames()).setOutputMode('RAW')
classified = covariates.clip(ecoregion.geometry()).classify(classifier).arrayFlatten([['natural', 'built', 'crops', 'mines', 'plantation']]).select('natural')

# pred_masks = classified.gte(1).eq(ee.Image.constant(ee.List.sequence(0, n_classes - 1)))
# true_masks = reclass.gte(1).eq(ee.Image.constant(ee.List.sequence(0, n_classes - 1)))
# eeprint(pred_masks)

In [ ]:
Map = geemap.Map()
Map.addLayer(classified)
Map.centerObject(ecoregion.geometry())
# Map.addLayer(true_masks)
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

In [ ]:
# for permutation importance
# compute baseline accuracy using all features
# for each feature, permute (shuffle) values

In [27]:
def iou(true_masks, pred_masks, clsno, grid_filtered):
    """
    Compute global IoU for a specific class across multiple tiles efficiently.

    Args:
        true_masks (ee.Image): ground truth mask image (multi-band)
        pred_masks (ee.Image): predicted mask image (multi-band)
        clsno (int): class index (band number)
        grid_filtered (ee.FeatureCollection): tiling grid

    Returns:
        ee.Number: IoU value for the specified class
    """
    # Select class band
    pred_mask = pred_masks.select([clsno])
    true_mask = true_masks.select([clsno])

    intersection = pred_mask.And(true_mask).selfMask()
    union = pred_mask.Or(true_mask).selfMask()

    # For each group, sum the number of pixels
    sums = intersection.addBands(union).rename(['intersection', 'union']).reduceRegions(
                collection= grid_filtered,
                reducer=ee.Reducer.count(),
                scale= pred_mask.projection().nominalScale(),
                maxPixelsPerRegion=1e9
            )

    # Aggregate totals across all tiles
    totals = sums.reduceColumns(
        reducer=ee.Reducer.sum().repeat(2),
        selectors=['intersection', 'union']
    )

    values = ee.List(totals.get('sum'))
    intersection_count = values.getNumber(0)
    union_count = values.getNumber(1)

    # Compute IoU from aggregated counts
    iou = intersection_count.divide(union_count)

    return ee.Feature(None, {'iou': iou, 'class': clsno})

In [31]:
for clsno in tqdm(range(2)):
    eeprint(iou(true_masks, pred_masks, clsno, grid_filtered.filterBounds(ecoregion.geometry())))

  0%|          | 0/2 [00:00<?, ?it/s]

EEException: User memory limit exceeded.

In [67]:
np.array([ 0.6430936695671063, 0.4125591742191372]).mean()

np.float64(0.5278264218931218)

In [68]:
grid_filtered.filterBounds(ecoregion.geometry()).size()

In [ ]:
baseline = ee.List.sequence(0, n_classes-1).map(
    lambda clsno: iou(true_masks, pred_masks, 0,
        grid_filtered.filterBounds(ecoregion.geometry())))
print('classwise means:')
eeprint(baseline)

# baseline_miou = ee.FeatureCollection(baseline).aggregate_mean('iou')
# print('Baseline MIoU:')
# eeprint(baseline_miou)

In [11]:
def permute_feature_in_training(train, feature_name):
    """
    Permute a feature by shuffling its values across the training samples.
    
    Args:
        train: ee.FeatureCollection - training data
        feature_name: str - name of feature to permute
    
    Returns:
        ee.FeatureCollection with the specified feature shuffled
    """
    # Extract the feature values as a list
    feature_values = train.aggregate_array(feature_name)
    
    # Shuffle the list
    shuffled_values = feature_values.shuffle(seed=42)
    
    # Create a list of indices
    indices = ee.List.sequence(0, train.size().subtract(1))
    
    # Convert FeatureCollection to list
    train_list = train.toList(train.size())
    
    # Map over indices to reassign shuffled values
    def reassign_feature(index):
        index = ee.Number(index)
        feature = ee.Feature(train_list.get(index))
        new_value = shuffled_values.get(index)
        return feature.set(feature_name, new_value)
    
    permuted_train = ee.FeatureCollection(indices.map(reassign_feature))
    
    return permuted_train

# Usage example:
permuted_train = permute_feature_in_training(train, 'structure')

# Retrain classifier with permuted data
classifier_permuted = ee.Classifier.minimumDistance(metric='mahalanobis').train(
    features=permuted_train,
    classProperty='first',
    inputProperties=['structure', 'function', 'compositio']
).setOutputMode('CLASSIFICATION')

# Classify and evaluate
classified_permuted = covariates.rename(['structure', 'function', 'compositio']).classify(classifier_permuted)
pred_masks_permuted = classified_permuted.eq(ee.Image.constant(ee.List.sequence(0, n_classes - 1)))

In [56]:
permuted_struc = ee.List.sequence(0, n_classes-1).map(
    lambda clsno: iou(true_masks, pred_masks_permuted, clsno,
        grid_filtered.filterBounds(ecoregion.geometry()).limit(5)))
print('classwise means:')
eeprint(permuted_struc)

miou = ee.FeatureCollection(permuted_struc).aggregate_mean('iou')
print('permuted structure MIoU:')
eeprint(miou)

classwise means:


permuted structure MIoU:


In [12]:
vars = ['structure', 'function', 'compositio']

pis = []
baseline_miou = 0.5278264218931218

for var in tqdm(vars):
    permuted_train = permute_feature_in_training(train, var)

    # Retrain classifier with permuted data
    classifier_permuted = ee.Classifier.minimumDistance(metric='mahalanobis').train(
        features=permuted_train,
        classProperty='first',
        inputProperties= ['structure', 'function', 'compositio']
    ).setOutputMode('CLASSIFICATION')

    # Classify and evaluate
    classified_permuted = covariates.rename(['structure', 'function', 'compositio']).classify(classifier_permuted)
    pred_masks_permuted = classified_permuted.eq(ee.Image.constant(ee.List.sequence(0, n_classes - 1)))

    class_iou = []
    for clsno in tqdm(range(2)):
        permuted_iou =  iou(true_masks, pred_masks_permuted, clsno,
        grid_filtered.filterBounds(ecoregion.geometry())).get('iou').getInfo()
        
        class_iou.append(permuted_iou)
    # print('classwise means:')
    # eeprint(permuted_struc)

    miou = np.array(class_iou).mean()

    result = {'feature': var, 'miou': miou, 'PI': baseline_miou-miou}
    pis.append(result)
    print(result)

# next, perform rfe by dropping least important feature, and performing pi, repeat until one feature remains. consider use of feature sets.

  0%|          | 0/3 [00:00<?, ?it/s]

  0%|          | 0/2 [00:00<?, ?it/s]

{'feature': 'structure', 'miou': np.float64(0.5098324892616548), 'PI': np.float64(0.01799393263146698)}


  0%|          | 0/2 [00:00<?, ?it/s]

{'feature': 'function', 'miou': np.float64(0.34737057027405427), 'PI': np.float64(0.18045585161906752)}


  0%|          | 0/2 [00:00<?, ?it/s]

{'feature': 'compositio', 'miou': np.float64(0.19254611174563657), 'PI': np.float64(0.3352803101474852)}


## Step 2: Train minimum distance classifier on all covariates

## Step 3: Predict closest class

## Step 4: compute iou and miou